# Unified Zeek TSV Preprocessing Notebook

This notebook standardizes preprocessing for Zeek-generated TSV files.

In [19]:
import sys

sys.path.insert(0, '..')

from __future__ import annotations
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from utils.utils import *

In [20]:
def normalize_label_name(label: str) -> str:
    label = str(label).strip()
    return (
        label.replace("-", "_")
        .replace(" ", "_")
        .replace("/", "_")
        .upper()
    )

def clean_numeric_columns(df: pd.DataFrame, numeric_cols: list[str]) -> pd.DataFrame:
    df = df.copy()
    cols = [col for col in numeric_cols if col in df.columns]

    df[cols] = (
        df[cols]
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
    )
    return df

def compute_rate_feature(df: pd.DataFrame) -> pd.DataFrame:
    duration = df["duration"]
    orig_pkt_rate = df["orig_pkts"] / duration
    orig_byte_rate = df["orig_bytes"] / duration
    df["orig_pkt_rate"] = orig_pkt_rate
    df["orig_byte_rate"] = orig_byte_rate
    return df

def compute_valid_tcp_handshake(df: pd.DataFrame) -> pd.DataFrame:
    proto_ok = df["proto"].eq("tcp")
    history_ok = df["history"].str.contains(
        r"(?=.*S)(?=.*h)(?=.*A)",
        regex=True,
        na=False
    )
    df["valid_tcp_handshake"] = (proto_ok & history_ok).astype(int)
    return df


def compute_valid_http_conn(df: pd.DataFrame) -> pd.DataFrame:
    service = df["service"].astype(str).str.lower()

    proto_ok = df["proto"].eq("tcp")
    http_ok = df["id.resp_p"].isin([80, 8080, 8000]) & service.eq("http")
    https_ok = df["id.resp_p"].isin([443, 8443]) & service.isin(["https", "ssl"])
    has_data = df["history"].str.contains(r"D", regex=True, na=False)

    df["valid_http_conn"] = (proto_ok & (http_ok | https_ok) & has_data).astype(int)
    return df


def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    print_section("Adding engineered features")
    print(f"input rows: {len(df):,}")

    df = clean_numeric_columns(df, MODEL_NUMERIC_FEATURES)
    df = compute_valid_tcp_handshake(df)
    df = compute_valid_http_conn(df)
    df = compute_rate_feature(df)
    df = clean_numeric_columns(df, ENGINEERED_FEATURES)

    print(f"output rows: {len(df):,}")
    print("engineered features:", ", ".join(ENGINEERED_FEATURES))
    return df

def drop_invalid_http_flood_rows(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    is_dos = df["label"] == "DOS_HTTP_FLOOD"
    not_tcp = df["proto"].astype(str).str.lower().ne("tcp")
    not_correct_service = ~df["service"].astype(str).str.lower().isin({"http", "https", "ssl"})

    remove_mask = is_dos & ( not_tcp |  not_correct_service)

    removed = remove_mask.sum()
    print(f"Removed {removed} DOS_HTTP_FLOOD rows")

    return df.loc[~remove_mask].reset_index(drop=True)

In [21]:
def load_and_prepare_dataset(
    datapath: str,
    target_labels: list[str]
) -> pd.DataFrame:
    print(f"Loading data from: {datapath}")
    df = pd.read_csv(datapath, delimiter="\t", on_bad_lines="skip")
    df.columns = df.columns.str.strip()

    print("Raw shape:", df.shape)

    df.drop_duplicates(inplace=True)
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    df["label"] = df["label"].astype(str).map(normalize_label_name)
    target_labels = [normalize_label_name(x) for x in target_labels]
    df = df[df["label"].isin(target_labels)].copy()

    df = drop_invalid_http_flood_rows(df)
    df = add_engineered_features(df)


    print("Processed shape:", df.shape)
    if "label" in df.columns:
        print(df["label"].value_counts())

    return df

In [22]:
TARGET_LABELS = ["BENIGN", "DOS_HTTP_FLOOD", "PORTSCAN", "XSS", "SQL_INJECTION", "BRUTE_FORCE"]

def load_cicids2017_data() -> pd.DataFrame:
    print("Loading CICIDS2017 data...")
    df_cicids2017_monday = load_and_prepare_dataset(
        datapath="../data/CICIDS2017/monday_labeled.tsv",
        target_labels=TARGET_LABELS
    )
    df_cicids2017_tuesday = load_and_prepare_dataset(
        datapath="../data/CICIDS2017/tuesday_labeled.tsv",
        target_labels=TARGET_LABELS
    )
    df_cicids2017_wednesday = load_and_prepare_dataset(
        datapath="../data/CICIDS2017/wednesday_labeled.tsv",
        target_labels=TARGET_LABELS
    )
    df_cicids2017_thursday = load_and_prepare_dataset(
        datapath="../data/CICIDS2017/thursday_labeled.tsv",
        target_labels=TARGET_LABELS
    )
    df_cicids2017_friday = load_and_prepare_dataset(
        datapath="../data/CICIDS2017/friday_labeled.tsv",
        target_labels=TARGET_LABELS
    )
    res = pd.concat([df_cicids2017_monday, df_cicids2017_tuesday, df_cicids2017_wednesday, df_cicids2017_thursday, df_cicids2017_friday], ignore_index=True)
    print("Processed shape:", res.shape)
    print(res["label"].value_counts())
    return res

def load_ciciot2023_data(type: str) -> pd.DataFrame:
    print(f"Loading CICIoT2023_{type} data...")
    res = load_and_prepare_dataset(
        datapath=f"../data/CICIoT2023/ciciot2023_labeled_{type}.tsv",
        target_labels=TARGET_LABELS
    )
    print("Processed shape:", res.shape)
    if "label" in res.columns:
        print(res["label"].value_counts())
    return res

In [23]:
def downsample_label(df: pd.DataFrame, label: str, frac: float) -> pd.DataFrame:
    label_df = df[df["label"] == label]
    df_other = df[df["label"] != label]
    label_df = label_df.sample(frac=frac, random_state=42)
    combined_df = pd.concat([label_df, df_other])
    return combined_df.reset_index(drop=True)

In [24]:
df_cicids2017 = load_cicids2017_data()

Loading CICIDS2017 data...
Loading data from: ../data/CICIDS2017/monday_labeled.tsv
Raw shape: (375432, 23)
Removed 0 DOS_HTTP_FLOOD rows

Adding engineered features
input rows: 375,432
output rows: 375,432
engineered features: orig_pkt_rate, orig_byte_rate, time_elapsed, valid_tcp_handshake, valid_http_conn, uniq_dst_ports, pkts_per_port, scan_duration, fail_ratio
Processed shape: (375432, 27)
label
BENIGN    375432
Name: count, dtype: int64
Loading data from: ../data/CICIDS2017/tuesday_labeled.tsv
Raw shape: (6971, 23)
Removed 0 DOS_HTTP_FLOOD rows

Adding engineered features
input rows: 6,971
output rows: 6,971
engineered features: orig_pkt_rate, orig_byte_rate, time_elapsed, valid_tcp_handshake, valid_http_conn, uniq_dst_ports, pkts_per_port, scan_duration, fail_ratio
Processed shape: (6971, 27)
label
BRUTE_FORCE    6971
Name: count, dtype: int64
Loading data from: ../data/CICIDS2017/wednesday_labeled.tsv
Raw shape: (191001, 23)
Removed 8102 DOS_HTTP_FLOOD rows

Adding engineered f

C:\Users\Rasmus\AppData\Local\Temp\ipykernel_23892\3221699632.py:6: DtypeWarning: Columns (0: duration, 1: orig_bytes, 2: resp_bytes) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(datapath, delimiter="\t", on_bad_lines="skip")


Raw shape: (256100, 23)
Removed 0 DOS_HTTP_FLOOD rows

Adding engineered features
input rows: 160,417
output rows: 160,417
engineered features: orig_pkt_rate, orig_byte_rate, time_elapsed, valid_tcp_handshake, valid_http_conn, uniq_dst_ports, pkts_per_port, scan_duration, fail_ratio
Processed shape: (160417, 27)
label
PORTSCAN    160417
Name: count, dtype: int64
Processed shape: (700226, 27)
label
BENIGN            375432
PORTSCAN          160417
DOS_HTTP_FLOOD    155349
BRUTE_FORCE         8338
XSS                  672
SQL_INJECTION         18
Name: count, dtype: int64


In [25]:
df_ciciot2023_good = load_ciciot2023_data("good")

Loading CICIoT2023_good data...
Loading data from: ../data/CICIoT2023/ciciot2023_labeled_good.tsv
Raw shape: (1901956, 23)
Removed 1297178 DOS_HTTP_FLOOD rows

Adding engineered features
input rows: 342,485
output rows: 342,485
engineered features: orig_pkt_rate, orig_byte_rate, time_elapsed, valid_tcp_handshake, valid_http_conn, uniq_dst_ports, pkts_per_port, scan_duration, fail_ratio
Processed shape: (342485, 27)
label
BENIGN            160231
DOS_HTTP_FLOOD    119782
PORTSCAN           45869
BRUTE_FORCE         7658
SQL_INJECTION       5598
XSS                 3347
Name: count, dtype: int64
Processed shape: (342485, 27)
label
BENIGN            160231
DOS_HTTP_FLOOD    119782
PORTSCAN           45869
BRUTE_FORCE         7658
SQL_INJECTION       5598
XSS                 3347
Name: count, dtype: int64


In [26]:
df_ciciot2023_bad = load_ciciot2023_data("bad")

Loading CICIoT2023_bad data...
Loading data from: ../data/CICIoT2023/ciciot2023_labeled_bad.tsv
Raw shape: (2083980, 23)
Removed 1363138 DOS_HTTP_FLOOD rows

Adding engineered features
input rows: 720,842
output rows: 720,842
engineered features: orig_pkt_rate, orig_byte_rate, time_elapsed, valid_tcp_handshake, valid_http_conn, uniq_dst_ports, pkts_per_port, scan_duration, fail_ratio
Processed shape: (720842, 27)
label
BENIGN            342255
PORTSCAN          216533
DOS_HTTP_FLOOD    145451
BRUTE_FORCE         7658
SQL_INJECTION       5598
XSS                 3347
Name: count, dtype: int64
Processed shape: (720842, 27)
label
BENIGN            342255
PORTSCAN          216533
DOS_HTTP_FLOOD    145451
BRUTE_FORCE         7658
SQL_INJECTION       5598
XSS                 3347
Name: count, dtype: int64


In [27]:
df_cicids2017.to_csv(f"cicids2017_preprocessed.tsv", sep='\t', index=False, header=True)

In [28]:
df_ciciot2023_good.to_csv(f"ciciot2023_preprocessed_good.tsv", sep='\t', index=False, header=True)

In [29]:
df_ciciot2023_bad.to_csv(f"ciciot2023_preprocessed_bad.tsv", sep='\t', index=False, header=True)

# Create small datasets

In [30]:
BIG_LABEL = ["BENIGN", "PORTSCAN", "DOS_HTTP_FLOOD"]

def downsample_labels(df, TARGET_LABELS, frac=0.1):
    for label in TARGET_LABELS:
        df = downsample_label(df, label=label, frac=frac)
    print(df["label"].value_counts())
    return df.reset_index(drop=True)

cicids_small = downsample_labels(df_cicids2017, BIG_LABEL, frac=0.1)
cicids_small.to_csv(f"cicids2017_preprocessed_small.tsv", sep="\t", index=False, header=True)
ciciot_small_good = downsample_labels(df_ciciot2023_good, BIG_LABEL, frac=0.1)
ciciot_small_good.to_csv(f"ciciot2023_preprocessed_small_good.tsv", sep="\t", index=False, header=True)
ciciot_small_bad = downsample_labels(df_ciciot2023_bad, BIG_LABEL, frac=0.1)
ciciot_small_bad.to_csv(f"ciciot2023_preprocessed_small_bad.tsv", sep="\t", index=False, header=True)

label
BENIGN            37543
PORTSCAN          16042
DOS_HTTP_FLOOD    15535
BRUTE_FORCE        8338
XSS                 672
SQL_INJECTION        18
Name: count, dtype: int64
label
BENIGN            16023
DOS_HTTP_FLOOD    11978
BRUTE_FORCE        7658
SQL_INJECTION      5598
PORTSCAN           4587
XSS                3347
Name: count, dtype: int64
label
BENIGN            34226
PORTSCAN          21653
DOS_HTTP_FLOOD    14545
BRUTE_FORCE        7658
SQL_INJECTION      5598
XSS                3347
Name: count, dtype: int64
